# 02 - Capa Silver: Validacion y Calidad de Datos

Lee la tabla Bronze y aplica reglas de calidad para crear la capa Silver.

**Validaciones:**
1. La imagen debe existir en el volumen
2. Coordenadas del poligono en rango [0, 1]
3. Bounding box con area > 0
4. Clase valida (dentro del rango de clases conocidas)
5. Poligono con al menos 3 puntos
6. Sin anotaciones duplicadas por imagen/objeto

**Salida:** `jgworkspaceclassic_catalog.infra_demo.cv_v2_silver`

In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -- Unity Catalog --
CATALOG = "jgworkspaceclassic_catalog"
SCHEMA = "infra_demo"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_silver"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/computer_vision"

print(f"Bronze: {BRONZE_TABLE}")
print(f"Silver: {SILVER_TABLE}")

Bronze: jgworkspaceclassic_catalog.infra_demo.cv_v2_bronze
Silver: jgworkspaceclassic_catalog.infra_demo.cv_v2_silver


In [0]:
df_bronze = spark.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Registros en Bronze: {total_bronze}")
display(df_bronze.limit(5))

Registros en Bronze: 189


image_id image_path label_path split class_id class_name object_index polygon_coords num_polygon_points bbox_x_min bbox_y_min bbox_x_max bbox_y_max image_size_bytes objects_in_image ingested_at source roboflow_project roboflow_version IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146 /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/images/IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146.jpg /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/labels/IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146.txt train 0 Captain 0 List(0.5046875, 0.2359375, 0.5109375, 0.2390625, 0.5359375, 0.2390625, 0.559375, 0.24375, 0.5921875, 0.246875, 0.6, 0.2515625, 0.6171875, 0.2546875, 0.6328125, 0.2609375, 0.6703125, 0.2859375, 0.684375, 0.3, 0.696875, 0.325, 0.6953125, 0.3265625, 0.69375, 0.365625, 0.690625, 0.371875, 0.690625, 0.3890625, 0.6921875, 0.390625, 0.690625, 0.4140625, 0.6875, 0.41875, 0.6859375, 0.4390625, 0.675, 0.471875, 0.6609375, 0.4953125, 0.6515625, 0.5046875, 0.6515625, 0.5109375, 0.665625, 0.5296875, 0.665625, 0.5375, 0.6578125, 0.5421875, 0.6453125, 0.54375, 0.634375, 0.5515625, 0.6328125, 0.55, 0.6140625, 0.55, 0.59375, 0.5421875, 0.5875, 0.5359375, 0.5859375, 0.528125, 0.5828125, 0.525, 0.5453125, 0.525, 0.5421875, 0.5234375, 0.5265625, 0.5390625, 0.51875, 0.5421875, 0.50625, 0.55625, 0.509375, 0.5609375, 0.521875, 0.5625, 0.5265625, 0.565625, 0.5328125, 0.575, 0.55625, 0.565625, 0.5703125, 0.5640625, 0.571875, 0.56875, 0.5703125, 0.5796875, 0.5578125, 0.60625, 0.5578125, 0.61875, 0.5546875, 0.6296875, 0.5578125, 0.6328125, 0.565625, 0.63125, 0.5703125, 0.6359375, 0.5703125, 0.659375, 0.5765625, 0.6671875, 0.5796875, 0.6765625, 0.575, 0.6875, 0.5765625, 0.69375, 0.5984375, 0.6890625, 0.625, 0.671875, 0.6296875, 0.671875, 0.6328125, 0.6765625, 0.628125, 0.6859375, 0.6046875, 0.7140625, 0.6015625, 0.721875, 0.5921875, 0.7296875, 0.54375, 0.7390625, 0.5359375, 0.7375, 0.5015625, 0.74375, 0.4953125, 0.7421875, 0.475, 0.74375, 0.45625, 0.7484375, 0.446875, 0.7484375, 0.43125, 0.75625, 0.4, 0.7578125, 0.3546875, 0.7703125, 0.3171875, 0.7671875, 0.3140625, 0.7640625, 0.303125, 0.7625, 0.3046875, 0.7546875, 0.321875, 0.7375, 0.321875, 0.7328125, 0.3046875, 0.725, 0.2828125, 0.71875, 0.2640625, 0.6953125, 0.2484375, 0.6875, 0.246875, 0.6859375, 0.2484375, 0.6796875, 0.2421875, 0.6625, 0.23125, 0.653125, 0.2265625, 0.6453125, 0.2265625, 0.634375, 0.2296875, 0.63125, 0.2375, 0.6296875, 0.2421875, 0.625, 0.23125, 0.6125, 0.2265625, 0.6015625, 0.228125, 0.5984375, 0.2296875, 0.596875, 0.25, 0.5984375, 0.2640625, 0.5796875, 0.2734375, 0.5609375, 0.2796875, 0.5546875, 0.2796875, 0.5515625, 0.29375, 0.5453125, 0.303125, 0.5453125, 0.3078125, 0.540625, 0.31875, 0.5359375, 0.321875, 0.5375, 0.3328125, 0.5359375, 0.3640625, 0.51875, 0.3671875, 0.5203125, 0.375, 0.5140625, 0.3625, 0.5078125, 0.3390625, 0.5046875, 0.321875, 0.4984375, 0.2953125, 0.4796875, 0.2859375, 0.4640625, 0.2828125, 0.45, 0.2765625, 0.4359375, 0.2765625, 0.4265625, 0.2734375, 0.41875, 0.275, 0.3671875, 0.2828125, 0.340625, 0.2828125, 0.3171875, 0.2859375, 0.296875, 0.296875, 0.278125, 0.315625, 0.2625, 0.3625, 0.240625, 0.4140625, 0.2328125, 0.4171875, 0.234375, 0.503125, 0.2359375) 132 0.2265625 0.2328125 0.696875 0.7703125 28305 1 2026-04-23T01:43:58.882Z roboflow toys-detection-bzeh7 2 IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/images/IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de.jpg /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/labels/IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de.txt train 0 Captain 0 List(0.425, 0.2484375, 0.4265625, 0.25, 0.4390625, 0.2484375, 0.446875, 0.2515625, 0.4515625, 0.25, 0.4671875, 0.2546875, 0.4734375, 0.253125, 0.534375, 0.2671875, 0.5421875, 0.271875, 0.5734375, 0.278125, 0.5828125, 0.2828125, 0.59375, 0.284375, 0.59

In [0]:
# Verificar existencia de imagenes en el driver
image_paths = [row.image_path for row in df_bronze.select("image_path").distinct().collect()]
valid_images = {p for p in image_paths if p and os.path.exists(p)}
missing_images = set(image_paths) - valid_images

if missing_images:
    print(f"ALERTA: {len(missing_images)} imagenes no encontradas:")
    for p in list(missing_images)[:5]:
        print(f"  - {p}")
else:
    print(f"OK: Todas las {len(valid_images)} imagenes existen en el volumen")

# Crear DataFrame con las imagenes validas y hacer JOIN (compatible con serverless)
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
df_valid_images = spark.createDataFrame(
    [(p, True) for p in valid_images],
    schema=StructType([
        StructField("_valid_image_path", StringType(), False),
        StructField("v_image_exists", BooleanType(), False),
    ])
)

OK: Todas las 175 imagenes existen en el volumen


In [0]:
df_silver = (
    df_bronze
    # V1: Imagen existe (via JOIN, compatible con serverless)
    .join(df_valid_images, df_bronze.image_path == df_valid_images._valid_image_path, "left")
    .withColumn("v_image_exists", F.coalesce(F.col("v_image_exists"), F.lit(False)))
    .drop("_valid_image_path")

    # V2: Coordenadas en rango [0, 1]
    .withColumn("v_coords_in_range",
        (F.col("bbox_x_min") >= 0.0) & (F.col("bbox_x_max") <= 1.0) &
        (F.col("bbox_y_min") >= 0.0) & (F.col("bbox_y_max") <= 1.0)
    )

    # V3: Bounding box con area > 0
    .withColumn("bbox_width", F.col("bbox_x_max") - F.col("bbox_x_min"))
    .withColumn("bbox_height", F.col("bbox_y_max") - F.col("bbox_y_min"))
    .withColumn("bbox_area", F.col("bbox_width") * F.col("bbox_height"))
    .withColumn("v_bbox_valid", F.col("bbox_area") > 0)

    # V4: Clase valida (0 o 1 para este dataset)
    .withColumn("v_class_valid", F.col("class_id").isin(0, 1))

    # V5: Minimo 3 puntos en el poligono
    .withColumn("v_polygon_valid", F.col("num_polygon_points") >= 3)

    # Flag global de calidad
    .withColumn("is_valid",
        F.col("v_image_exists") &
        F.col("v_coords_in_range") &
        F.col("v_bbox_valid") &
        F.col("v_class_valid") &
        F.col("v_polygon_valid")
    )
    .withColumn("validated_at", F.current_timestamp())
)

In [0]:
# Verificar si hay anotaciones duplicadas (misma imagen, mismo indice de objeto)
w = Window.partitionBy("image_id", "object_index").orderBy("ingested_at")
df_silver = (
    df_silver
    .withColumn("_row_num", F.row_number().over(w))
    .withColumn("v_not_duplicate", F.col("_row_num") == 1)
    .drop("_row_num")
    # Actualizar is_valid incluyendo duplicados
    .withColumn("is_valid", F.col("is_valid") & F.col("v_not_duplicate"))
)

n_dupes = df_silver.filter(~F.col("v_not_duplicate")).count()
print(f"Duplicados detectados: {n_dupes}")

Duplicados detectados: 0


In [0]:
total = df_silver.count()
valid_count = df_silver.filter(F.col("is_valid")).count()
invalid_count = total - valid_count

print("=" * 60)
print("REPORTE DE CALIDAD DE DATOS")
print("=" * 60)
print(f"  Total registros:   {total}")
print(f"  Validos:           {valid_count} ({valid_count/total:.1%})")
print(f"  Invalidos:         {invalid_count} ({invalid_count/total:.1%})")
print()

checks = [
    ("v_image_exists", "Imagen existe en volumen"),
    ("v_coords_in_range", "Coordenadas en rango [0,1]"),
    ("v_bbox_valid", "Bounding box con area > 0"),
    ("v_class_valid", "Clase valida"),
    ("v_polygon_valid", "Poligono >= 3 puntos"),
    ("v_not_duplicate", "Sin duplicados"),
]

for col_name, desc in checks:
    passed = df_silver.filter(F.col(col_name)).count()
    failed = total - passed
    status = "PASS" if failed == 0 else "FAIL"
    print(f"  [{status}] {desc}: {passed}/{total} ({failed} fallaron)")

REPORTE DE CALIDAD DE DATOS
  Total registros:   189
  Validos:           189 (100.0%)
  Invalidos:         0 (0.0%)

  [PASS] Imagen existe en volumen: 189/189 (0 fallaron)
  [PASS] Coordenadas en rango [0,1]: 189/189 (0 fallaron)
  [PASS] Bounding box con area > 0: 189/189 (0 fallaron)
  [PASS] Clase valida: 189/189 (0 fallaron)
  [PASS] Poligono >= 3 puntos: 189/189 (0 fallaron)
  [PASS] Sin duplicados: 189/189 (0 fallaron)


In [0]:
display(
    df_silver
    .filter(F.col("is_valid"))
    .groupBy("split", "class_name")
    .agg(
        F.count("*").alias("anotaciones"),
        F.countDistinct("image_id").alias("imagenes"),
        F.avg("bbox_area").cast("decimal(6,4)").alias("avg_bbox_area"),
    )
    .orderBy("split", "class_name")
)

split,class_name,anotaciones,imagenes,avg_bbox_area
test,Captain,2,2,0.2356
test,Gohan,6,6,0.3308
train,Captain,108,108,0.2830
train,Gohan,57,57,0.3109
valid,Captain,10,10,0.2639
valid,Gohan,6,6,0.3445


In [0]:
df_silver.write.mode("overwrite").saveAsTable(SILVER_TABLE)
print(f"Tabla Silver guardada: {SILVER_TABLE}")
print(f"Total registros: {spark.table(SILVER_TABLE).count()}")
print(f"Registros validos: {spark.table(SILVER_TABLE).filter(F.col('is_valid')).count()}")

Tabla Silver guardada: jgworkspaceclassic_catalog.infra_demo.cv_v2_silver
Total registros: 189
Registros validos: 189


In [0]:
display(spark.table(SILVER_TABLE).limit(10))

image_id image_path label_path split class_id class_name object_index polygon_coords num_polygon_points bbox_x_min bbox_y_min bbox_x_max bbox_y_max image_size_bytes objects_in_image ingested_at source roboflow_project roboflow_version v_image_exists v_coords_in_range bbox_width bbox_height bbox_area v_bbox_valid v_class_valid v_polygon_valid is_valid validated_at v_not_duplicate IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146 /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/images/IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146.jpg /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/labels/IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146.txt train 0 Captain 0 List(0.5046875, 0.2359375, 0.5109375, 0.2390625, 0.5359375, 0.2390625, 0.559375, 0.24375, 0.5921875, 0.246875, 0.6, 0.2515625, 0.6171875, 0.2546875, 0.6328125, 0.2609375, 0.6703125, 0.2859375, 0.684375, 0.3, 0.696875, 0.325, 0.6953125, 0.3265625, 0.69375, 0.365625, 0.690625, 0.371875, 0.690625, 0.3890625, 0.6921875, 0.390625, 0.690625, 0.4140625, 0.6875, 0.41875, 0.6859375, 0.4390625, 0.675, 0.471875, 0.6609375, 0.4953125, 0.6515625, 0.5046875, 0.6515625, 0.5109375, 0.665625, 0.5296875, 0.665625, 0.5375, 0.6578125, 0.5421875, 0.6453125, 0.54375, 0.634375, 0.5515625, 0.6328125, 0.55, 0.6140625, 0.55, 0.59375, 0.5421875, 0.5875, 0.5359375, 0.5859375, 0.528125, 0.5828125, 0.525, 0.5453125, 0.525, 0.5421875, 0.5234375, 0.5265625, 0.5390625, 0.51875, 0.5421875, 0.50625, 0.55625, 0.509375, 0.5609375, 0.521875, 0.5625, 0.5265625, 0.565625, 0.5328125, 0.575, 0.55625, 0.565625, 0.5703125, 0.5640625, 0.571875, 0.56875, 0.5703125, 0.5796875, 0.5578125, 0.60625, 0.5578125, 0.61875, 0.5546875, 0.6296875, 0.5578125, 0.6328125, 0.565625, 0.63125, 0.5703125, 0.6359375, 0.5703125, 0.659375, 0.5765625, 0.6671875, 0.5796875, 0.6765625, 0.575, 0.6875, 0.5765625, 0.69375, 0.5984375, 0.6890625, 0.625, 0.671875, 0.6296875, 0.671875, 0.6328125, 0.6765625, 0.628125, 0.6859375, 0.6046875, 0.7140625, 0.6015625, 0.721875, 0.5921875, 0.7296875, 0.54375, 0.7390625, 0.5359375, 0.7375, 0.5015625, 0.74375, 0.4953125, 0.7421875, 0.475, 0.74375, 0.45625, 0.7484375, 0.446875, 0.7484375, 0.43125, 0.75625, 0.4, 0.7578125, 0.3546875, 0.7703125, 0.3171875, 0.7671875, 0.3140625, 0.7640625, 0.303125, 0.7625, 0.3046875, 0.7546875, 0.321875, 0.7375, 0.321875, 0.7328125, 0.3046875, 0.725, 0.2828125, 0.71875, 0.2640625, 0.6953125, 0.2484375, 0.6875, 0.246875, 0.6859375, 0.2484375, 0.6796875, 0.2421875, 0.6625, 0.23125, 0.653125, 0.2265625, 0.6453125, 0.2265625, 0.634375, 0.2296875, 0.63125, 0.2375, 0.6296875, 0.2421875, 0.625, 0.23125, 0.6125, 0.2265625, 0.6015625, 0.228125, 0.5984375, 0.2296875, 0.596875, 0.25, 0.5984375, 0.2640625, 0.5796875, 0.2734375, 0.5609375, 0.2796875, 0.5546875, 0.2796875, 0.5515625, 0.29375, 0.5453125, 0.303125, 0.5453125, 0.3078125, 0.540625, 0.31875, 0.5359375, 0.321875, 0.5375, 0.3328125, 0.5359375, 0.3640625, 0.51875, 0.3671875, 0.5203125, 0.375, 0.5140625, 0.3625, 0.5078125, 0.3390625, 0.5046875, 0.321875, 0.4984375, 0.2953125, 0.4796875, 0.2859375, 0.4640625, 0.2828125, 0.45, 0.2765625, 0.4359375, 0.2765625, 0.4265625, 0.2734375, 0.41875, 0.275, 0.3671875, 0.2828125, 0.340625, 0.2828125, 0.3171875, 0.2859375, 0.296875, 0.296875, 0.278125, 0.315625, 0.2625, 0.3625, 0.240625, 0.4140625, 0.2328125, 0.4171875, 0.234375, 0.503125, 0.2359375) 132 0.2265625 0.2328125 0.696875 0.7703125 28305 1 2026-04-23T01:43:58.882Z roboflow toys-detection-bzeh7 2 true true 0.47031248 0.5375 0.25279295 true true true true 2026-04-23T16:12:24.436Z true IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/images/IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de.jpg /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/labels/IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de.txt train 0 Captain 0 List(0.425, 0.2